In [ ]:
# Importar librerías estándar para análisis de datos y visualización
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Herramientas de Scikit-learn para métricas de clasificación
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# TensorFlow / Keras: API Sequential, capas convolucionales, regularización y utilidades
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout, Conv2D, MaxPool2D, Flatten, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical

In [ ]:
# Dataset MNIST incluido en Keras
from tensorflow.keras.datasets import mnist

In [ ]:
# Fijar semillas para reproducibilidad de resultados
import tensorflow as tf
np.random.seed(42)
tf.random.set_seed(42)

### Análisis Exploratorio y Visualización

In [ ]:
# Cargar el dataset MNIST: 60,000 imágenes de entrenamiento y 10,000 de prueba
(X_train, y_train), (X_test, y_test) = mnist.load_data()

In [ ]:
# Forma del conjunto de entrenamiento: (muestras, alto, ancho)
X_train.shape

In [ ]:
# Extraer una única imagen para inspeccionarla
single_image = X_train[0]

In [ ]:
# Dimensiones de una imagen individual: 28x28 píxeles en escala de grises
single_image.shape

In [ ]:
# Visualizar la imagen seleccionada
plt.imshow(single_image, cmap='gray');

### Etiquetado

In [ ]:
# Valores de las etiquetas en el conjunto de entrenamiento (enteros del 0 al 9)
y_train

In [ ]:
# Forma del vector de etiquetas
y_train.shape

In [ ]:
# Convertir etiquetas enteras a codificación one-hot (requerido por categorical_crossentropy)
# Si se usara sparse_categorical_crossentropy, este paso no sería necesario
y_categorical_train = to_categorical(y_train, 10)
y_categorical_test = to_categorical(y_test, 10)

In [ ]:
# Verificar la representación one-hot de las etiquetas de entrenamiento
y_categorical_train

In [ ]:
# Verificar la representación one-hot de las etiquetas de prueba
y_categorical_test

In [ ]:
# Forma del tensor one-hot: (muestras, número de clases)
y_categorical_train.shape

### Escalado y Reshaping

In [ ]:
# Valor máximo de los píxeles en MNIST (escala de grises: 0-255)
X_train.max()

In [ ]:
# Normalizar los píxeles al rango [0, 1] dividiendo por 255.0
# Usamos 255.0 explícito porque es el valor máximo teórico del dataset MNIST
X_train = X_train / 255.0
X_test = X_test / 255.0

In [ ]:
# Verificar la imagen normalizada (ahora los valores están entre 0 y 1)
plt.imshow(X_train[0], cmap='gray');

In [ ]:
# Forma actual del conjunto de entrenamiento antes de agregar el canal
X_train.shape

In [ ]:
# Forma actual del conjunto de prueba antes de agregar el canal
X_test.shape

In [ ]:
# Agregar la dimensión del canal (escala de grises = 1) para que sea compatible con Conv2D
# Las CNNs de Keras esperan entrada de forma: (batch, alto, ancho, canales)
X_train = X_train.reshape(X_train.shape[0], 28, 28, 1)
X_test = X_test.reshape(X_test.shape[0], 28, 28, 1)

In [ ]:
# Forma final del conjunto de entrenamiento: (muestras, 28, 28, 1)
X_train.shape

In [ ]:
# Forma final del conjunto de prueba: (muestras, 28, 28, 1)
X_test.shape

### Entrenar Modelo

In [ ]:
# Definir arquitectura de la CNN moderna con capas convolucionales, BatchNorm y Dropout
model = Sequential([
    Input(shape=(28, 28, 1)),
    
    # Bloque convolucional 1: extracción de características de bajo nivel (bordes, texturas)
    Conv2D(filters=32, kernel_size=(3, 3), activation='relu', padding='same'),
    BatchNormalization(),                             # Normaliza activaciones para acelerar convergencia
    MaxPool2D(pool_size=(2, 2)),                      # Reduce dimensiones espaciales a la mitad
    Dropout(0.3),                                     # Apaga 30% de neuronas para reducir sobreajuste
    
    # Bloque convolucional 2: extracción de características de mayor nivel
    Conv2D(filters=64, kernel_size=(3, 3), activation='relu', padding='same'),
    BatchNormalization(),
    MaxPool2D(pool_size=(2, 2)),
    Dropout(0.3),
    
    # Aplanar la salida 2D en un vector 1D para las capas densas
    Flatten(),
    
    # Capa densa oculta con alta capacidad de representación
    Dense(units=256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),                                     # Dropout más agresivo en capas densas (más propensas a overfitting)
    
    # Capa de salida: 10 neuronas (una por dígito) con softmax para probabilidades de clase
    Dense(units=10, activation='softmax')
])

**Nota sobre funciones de pérdida para clasificación multiclase:**

- **Categorical Crossentropy**: espera etiquetas en formato *one-hot* (matriz de forma `(muestras, clases)`).
- **Sparse Categorical Crossentropy**: espera etiquetas como enteros (vector de forma `(muestras,)`). Es más eficiente en memoria y no requiere `to_categorical`.

Ambas producen el mismo resultado; la elección depende del formato de las etiquetas.

In [ ]:
# Compilar el modelo: categorical_crossentropy para etiquetas one-hot
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# EarlyStopping: detiene el entrenamiento si val_loss no mejora en 3 épocas consecutivas
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

In [ ]:
# Entrenar el modelo con callback de EarlyStopping para evitar sobreajuste
history = model.fit(
    x=X_train,
    y=y_categorical_train,
    validation_data=(X_test, y_categorical_test),
    callbacks=[early_stop],
    batch_size=128,
    epochs=30
)

### Evaluación

In [ ]:
# Convertir el historial de entrenamiento a DataFrame para graficar
history_df = pd.DataFrame(history.history)

In [ ]:
# Graficar la evolución de la accuracy en entrenamiento y validación
history_df[['accuracy', 'val_accuracy']].plot();

In [ ]:
# Graficar la evolución de la pérdida en entrenamiento y validación
history_df[['loss', 'val_loss']].plot();

In [ ]:
# Evaluar el modelo en el conjunto de prueba y mostrar loss y accuracy finales
loss, accuracy = model.evaluate(X_test, y_categorical_test, verbose=0)
print(f'Loss en test: {loss:.4f}')
print(f'Accuracy en test: {accuracy:.4f}')

In [ ]:
# Predecir clases para todo el conjunto de prueba
# np.argmax convierte el vector de probabilidades (one-hot) al índice de la clase más probable
predictions = np.argmax(model.predict(X_test), axis=1)
predictions

In [ ]:
# Etiquetas reales del conjunto de prueba (en formato entero, compatible con predictions)
y_test

In [ ]:
# Reporte de clasificación: precision, recall y f1-score por dígito
print(classification_report(y_test, predictions))

In [ ]:
# Matriz de confusión: visualizar aciertos y errores por clase
plt.figure(figsize=(12, 12))
sns.heatmap(confusion_matrix(y_test, predictions), annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.title('Matriz de Confusión - MNIST');

### Predecir Clases

In [ ]:
# Seleccionar una imagen individual del conjunto de prueba para inferencia
my_number = X_test[100]
plt.imshow(my_number.reshape(28, 28), cmap='gray')
plt.title('Imagen de entrada para predicción');

In [ ]:
# Realizar predicción sobre la imagen individual
# Se agrega la dimensión batch para que sea compatible con el modelo: (1, 28, 28, 1)
prediction = np.argmax(model.predict(my_number.reshape(1, 28, 28, 1)), axis=1)
print(f'El modelo predice que el dígito es: {prediction[0]}')